# ⚡ Virtual Power Plant (VPP) — Production RL Baseline Training & Benchmarking

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/)

This Google Colab notebook trains production-grade **Proximal Policy Optimization (PPO)** agents for multi-objective Virtual Power Plant control across 5 benchmark scenarios:
1. **`easy-arbitrage`** — Electricity price arbitrage
2. **`carbon-minimal`** — Carbon intensity minimization
3. **`peak-shaving`** — Peak demand shaving & grid relief
4. **`degraded-battery`** — State-of-Health preservation under battery degradation
5. **`grid-stress-test`** — High volatility emergency grid response

---

## 1. Environment Setup & Dependencies

In [ ]:
# Install Gymnasium and Stable-Baselines3
!pip install -q gymnasium stable-baselines3 pandas numpy matplotlib tabulate
import torch
print(f"✅ PyTorch version: {torch.__version__} (CUDA available: {torch.cuda.is_available()})")


## 2. Define Core VPP Environment & Gymnasium Wrapper
*(Embedded self-contained VPP environment logic)*

In [ ]:
import numpy as np
import pandas as pd
import math
from typing import Dict, List, Tuple, Any, Optional
import gymnasium as gym
from gymnasium import spaces

class VppEnvironmentColab:
    """Production multi-objective VPP environment simulation engine."""
    def __init__(self, task_id: str = 'easy-arbitrage', max_steps: int = 96, seed: Optional[int] = None):
        self.task_id = task_id
        self.max_steps = max_steps
        self.current_step = 0
        self.np_random = np.random.RandomState(seed if seed is not None else 42)
        self.time_horizon = 24
        self._init_profiles()
        
    def _init_profiles(self):
        steps = self.max_steps
        t = np.linspace(0, 24, steps)
        solar_base = np.maximum(0, np.sin((t - 6) * np.pi / 12))
        solar_noise = self.np_random.normal(0, 0.05, steps)
        self.solar_profile = np.clip(solar_base + solar_noise, 0, 1) * 50.0
        demand_base = 20.0 + 15.0 * np.sin((t - 7) * np.pi / 12)**2 + 25.0 * np.sin((t - 18) * np.pi / 6)**2
        self.demand_profile = np.clip(demand_base + self.np_random.normal(0, 2, steps), 5, 80)
        price_base = 0.10 + 0.15 * (np.sin((t - 17) * np.pi / 6)**2) + 0.05 * (np.sin((t - 8) * np.pi / 12)**2)
        if self.task_id == 'grid-stress-test':
            price_base += self.np_random.choice([0, 0.40], size=steps, p=[0.85, 0.15])
        self.price_profile = np.clip(price_base, 0.04, 0.60)
        carbon_base = 300 - 150 * solar_base / (solar_base.max() + 1e-5) + 100 * (price_base / price_base.max())
        self.carbon_profile = np.clip(carbon_base + self.np_random.normal(0, 15, steps), 50, 600)
        self.soc = 0.5
        self.soh = 1.0
        self.battery_capacity_kwh = 100.0
        self.max_power_kw = 50.0
        
    def reset(self, seed: Optional[int] = None) -> Tuple[np.ndarray, Dict]:
        if seed is not None:
            self.np_random = np.random.RandomState(seed)
        self.current_step = 0
        self._init_profiles()
        return self.get_observation(), {}

    def get_observation(self) -> np.ndarray:
        t_idx = min(self.current_step, self.max_steps - 1)
        obs = np.zeros(52, dtype=np.float32)
        obs[0] = (self.current_step % 4) / 4.0
        obs[1] = (self.current_step // 4) / 24.0
        obs[2] = 0.0
        for i in range(9):
            idx = min(t_idx + i, self.max_steps - 1)
            obs[3 + i] = self.solar_profile[idx] / 50.0
        for i in range(9):
            idx = min(t_idx + i, self.max_steps - 1)
            obs[12 + i] = self.demand_profile[idx] / 100.0
        for i in range(9):
            idx = min(t_idx + i, self.max_steps - 1)
            obs[21 + i] = self.price_profile[idx] / 0.60
        for i in range(9):
            idx = min(t_idx + i, self.max_steps - 1)
            obs[30 + i] = self.carbon_profile[idx] / 600.0
        obs[39] = self.soc
        obs[40] = self.soh
        obs[41] = 0.0
        obs[42] = 25.0 / 45.0
        obs[43] = 1.0
        obs[44] = 1.0
        obs[45] = (self.demand_profile[t_idx] - self.solar_profile[t_idx]) / 100.0
        obs[46:52] = 0.05
        return obs

    def step(self, action: np.ndarray) -> Tuple[np.ndarray, float, bool, bool, Dict]:
        t_idx = self.current_step
        battery_action = float(np.clip(action[0], -1.0, 1.0))
        p_solar = self.solar_profile[t_idx]
        p_demand = self.demand_profile[t_idx]
        price = self.price_profile[t_idx]
        carbon = self.carbon_profile[t_idx]
        p_batt_requested = battery_action * self.max_power_kw
        dt = 0.25
        eff = 0.95
        if p_batt_requested > 0:
            max_discharge = (self.soc - 0.10) * self.battery_capacity_kwh / dt * eff
            p_batt = min(p_batt_requested, max_discharge)
            self.soc -= (p_batt / eff * dt) / self.battery_capacity_kwh
        else:
            max_charge = (0.90 - self.soc) * self.battery_capacity_kwh / dt / eff
            p_batt = max(p_batt_requested, -max_charge)
            self.soc -= (p_batt * eff * dt) / self.battery_capacity_kwh
        self.soc = float(np.clip(self.soc, 0.05, 0.95))
        throughput_kwh = abs(p_batt) * dt
        deg_cost = throughput_kwh * 0.02
        self.soh -= (throughput_kwh / (self.battery_capacity_kwh * 3000))
        p_net = p_demand - p_solar - p_batt
        energy_cost = p_net * price * dt
        carbon_emissions = max(0, p_net) * carbon * dt
        if self.task_id == 'easy-arbitrage':
            reward = -energy_cost * 1.5 - deg_cost
        elif self.task_id == 'carbon-minimal':
            reward = - (carbon_emissions / 1000.0) * 2.0 - energy_cost * 0.5 - deg_cost
        elif self.task_id == 'peak-shaving':
            peak_penalty = max(0, p_net - 30.0)**2 * 0.05
            reward = -energy_cost - peak_penalty - deg_cost
        elif self.task_id == 'degraded-battery':
            reward = -energy_cost - deg_cost * 5.0
        else:
            stress_penalty = max(0, p_net - 25.0)**2 * 0.10
            reward = -energy_cost * 2.0 - stress_penalty - deg_cost
        self.current_step += 1
        done = self.current_step >= self.max_steps
        info = {
            'energy_cost': energy_cost,
            'carbon_emissions': carbon_emissions,
            'p_net': p_net,
            'soc': self.soc,
            'soh': self.soh
        }
        return self.get_observation(), reward, done, False, info

class VppGymEnv(gym.Env):
    """Gymnasium wrapper for SB3 compliance."""
    metadata = {"render_modes": []}
    def __init__(self, task_id: str = 'easy-arbitrage', max_steps: int = 96, seed: Optional[int] = None):
        super().__init__()
        self.task_id = task_id
        self.max_steps = max_steps
        self._seed = seed
        self.env = VppEnvironmentColab(task_id=task_id, max_steps=max_steps, seed=seed)
        self.observation_space = spaces.Box(low=-np.inf, high=np.inf, shape=(52,), dtype=np.float32)
        self.action_space = spaces.Box(low=-1.0, high=1.0, shape=(5,), dtype=np.float32)

    def reset(self, seed: Optional[int] = None, options: Optional[Dict] = None):
        super().reset(seed=seed)
        effective_seed = seed if seed is not None else self._seed
        obs, info = self.env.reset(seed=effective_seed)
        return obs.astype(np.float32), info

    def step(self, action: np.ndarray):
        obs, reward, done, truncated, info = self.env.step(action)
        return obs.astype(np.float32), float(reward), done, truncated, info

print("✅ Core VPP Gymnasium Environment defined!")


## 3. Train Production PPO Agents with VecNormalize across All 5 Tasks

In [ ]:
import os
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize

TASKS = ['easy-arbitrage', 'carbon-minimal', 'peak-shaving', 'degraded-battery', 'grid-stress-test']
TOTAL_TIMESTEPS = 150000

os.makedirs('trained_models', exist_ok=True)
trained_models = {}
vec_envs = {}

for task in TASKS:
    print(f"\n" + "="*60)
    print(f"🚀 Training PPO Agent for task: {task} ({TOTAL_TIMESTEPS:,} steps)")
    print("="*60)
    def make_env():
        return VppGymEnv(task_id=task, seed=42)
    vec_env = DummyVecEnv([make_env])
    vec_env = VecNormalize(vec_env, norm_obs=True, norm_reward=True, clip_obs=10.0)
    model = PPO(
        policy='MlpPolicy',
        env=vec_env,
        learning_rate=3e-4,
        n_steps=2048,
        batch_size=64,
        n_epochs=10,
        gamma=0.99,
        gae_lambda=0.95,
        clip_range=0.2,
        ent_coef=0.005,
        verbose=0
    )
    model.learn(total_timesteps=TOTAL_TIMESTEPS)
    model_path = os.path.join('trained_models', f"ppo_{task}.zip")
    vec_path = os.path.join('trained_models', f"ppo_{task}_vecnormalize.pkl")
    model.save(model_path)
    vec_env.save(vec_path)
    trained_models[task] = model
    vec_envs[task] = vec_env
    print(f"✅ Successfully trained & saved: {model_path} + {vec_path}")

print("\n🎉 ALL 5 PPO AGENTS TRAINED SUCCESSFULLY!")


## 4. Head-to-Head Evaluation: Trained PPO vs Rule-Based Baseline
*(Evaluated on 5 disjoint test seeds: 1000-1004)*

In [ ]:
from tabulate import tabulate
import numpy as np

def rule_based_action(obs: np.ndarray) -> np.ndarray:
    price = obs[21] * 0.60
    solar = obs[3] * 50.0
    soc = obs[39]
    if price < 0.12 or solar > 30.0:
        action_batt = -0.8
    elif price > 0.25 and soc > 0.20:
        action_batt = 0.8
    else:
        action_batt = 0.0
    return np.array([action_batt, 0, 0, 0, 0], dtype=np.float32)

eval_seeds = [1000, 1001, 1002, 1003, 1004]
results = []

for task in TASKS:
    model = trained_models[task]
    vec_env = vec_envs[task]
    ppo_rewards, ppo_costs, ppo_carbons = [], [], []
    for seed in eval_seeds:
        eval_env = VppGymEnv(task_id=task, seed=seed)
        obs, _ = eval_env.reset(seed=seed)
        done = False
        ep_reward, ep_cost, ep_carbon = 0, 0, 0
        while not done:
            norm_obs = vec_env.normalize_obs(obs)
            action, _ = model.predict(norm_obs, deterministic=True)
            obs, r, done, _, info = eval_env.step(action)
            ep_reward += r
            ep_cost += info['energy_cost']
            ep_carbon += info['carbon_emissions']
        ppo_rewards.append(ep_reward)
        ppo_costs.append(ep_cost)
        ppo_carbons.append(ep_carbon)
    rb_rewards, rb_costs, rb_carbons = [], [], []
    for seed in eval_seeds:
        eval_env = VppGymEnv(task_id=task, seed=seed)
        obs, _ = eval_env.reset(seed=seed)
        done = False
        ep_reward, ep_cost, ep_carbon = 0, 0, 0
        while not done:
            action = rule_based_action(obs)
            obs, r, done, _, info = eval_env.step(action)
            ep_reward += r
            ep_cost += info['energy_cost']
            ep_carbon += info['carbon_emissions']
        rb_rewards.append(ep_reward)
        rb_costs.append(ep_cost)
        rb_carbons.append(ep_carbon)
    avg_ppo_r, avg_rb_r = np.mean(ppo_rewards), np.mean(rb_rewards)
    avg_ppo_cost, avg_rb_cost = np.mean(ppo_costs), np.mean(rb_costs)
    avg_ppo_carb, avg_rb_carb = np.mean(ppo_carbons), np.mean(rb_carbons)
    imprv = ((avg_ppo_r - avg_rb_r) / abs(avg_rb_r)) * 100.0
    results.append([
        task,
        f"${avg_rb_cost:.2f}", f"${avg_ppo_cost:.2f}",
        f"{avg_rb_carb/1000:.1f} kg", f"{avg_ppo_carb/1000:.1f} kg",
        f"{avg_rb_r:.1f}", f"{avg_ppo_r:.1f}",
        f"{imprv:+.1f}%",
    ])

headers = ["Task Scenario", "Rule Cost", "PPO Cost", "Rule Carbon", "PPO Carbon", "Rule Reward", "PPO Reward", "PPO vs Rule"]
print("\n" + "="*90)
print("📊 HEAD-TO-HEAD BENCHMARK EVALUATION RESULTS (Disjoint Eval Seeds 1000-1004)")
print("="*90)
print(tabulate(results, headers=headers, tablefmt="fancy_grid"))


## 5. Download Trained Models & VecNormalize Stats
Run this cell to download `vpp_trained_models.zip`. Unpack its contents into your local repository's `backend/trained_models/` folder!

In [ ]:
!zip -r vpp_trained_models.zip trained_models/
from google.colab import files
files.download('vpp_trained_models.zip')
print("📦 vpp_trained_models.zip download started!")
